# Fun-ASR-Nano with native Transformers

Chinese, English and Japanese transcription, without the FunASR toolkit or remote Python code. This notebook uses CPU float32, pinned Transformers 5.17.0 and the official native checkpoint. The first run downloads about 1.66 GB of weights. Colab runtimes change: verify dependency output before loading a model.

[Model card](https://huggingface.co/FunAudioLLM/Fun-ASR-Nano-2512-hf) | [Guide](https://www.funasr.com/en/docs/native-transformers.html) | [Fun-ASR](https://github.com/QwenAudio/Fun-ASR)


In [ ]:
%pip install --index-url https://download.pytorch.org/whl/cpu 'torch==2.10.0+cpu' 'torchaudio==2.10.0+cpu'
%pip install 'transformers==5.17.0' 'librosa==0.11.0' 'soundfile==0.13.1'
%pip check


## Transcribe the official English sample
Restart the notebook session after installation if it had already imported Torch or Transformers. Then run the following cell. This uses the original model repository only for its public audio sample; inference weights come from the separate native `-hf` checkpoint.


In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

torch.set_num_threads(4)
model_id = "FunAudioLLM/Fun-ASR-Nano-2512-hf"
revision = "d93b302ee7fd505e1b3576120fc142fc6f7820e1"
audio = "https://huggingface.co/FunAudioLLM/Fun-ASR-Nano-2512/resolve/272c57b82523ada6fd87095e955f8e29100979ab/example/en.mp3"

processor = AutoProcessor.from_pretrained(
    model_id, revision=revision, trust_remote_code=False, token=False
)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, revision=revision, trust_remote_code=False, token=False,
    dtype=torch.float32,
).to("cpu").eval()
inputs = processor.apply_transcription_request(
    audio=audio, language="en",
    processor_kwargs={
        "return_tensors": "pt",
        "audio_kwargs": {"sampling_rate": 16000},
        "text_kwargs": {"padding": True},
    },
)
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=128, do_sample=False)
new_tokens = generated[:, inputs.input_ids.shape[1]:]
print(processor.batch_decode(new_tokens, skip_special_tokens=True)[0])


## Transcribe an uploaded recording
Upload a short recording you are authorized to process to the notebook's Files panel. Set its path below. Audio is averaged to mono and explicitly resampled to 16 kHz; it is not renamed or trimmed. This short-file example rejects audio longer than 60 seconds. Use `zh`, `en` or `ja` to match the recording.


In [ ]:
import librosa
import numpy as np
import soundfile as sf

path = "/content/audio.wav"
info = sf.info(path)
if not 0 < info.duration <= 60:
    raise ValueError("Use a non-empty recording of at most 60 seconds")
waveform, sample_rate = sf.read(path, dtype="float32")
if not np.isfinite(waveform).all():
    raise ValueError("Non-finite audio samples")
if waveform.ndim == 2:
    waveform = waveform.mean(axis=1)
if sample_rate != 16000:
    waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=16000)
inputs = processor.apply_transcription_request(
    audio=waveform, language="zh",
    processor_kwargs={"return_tensors": "pt", "audio_kwargs": {"sampling_rate": 16000}},
)
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=256, do_sample=False)
new_tokens = generated[:, inputs.input_ids.shape[1]:]
print(processor.batch_decode(new_tokens, skip_special_tokens=True)[0])
eos = model.generation_config.eos_token_id
eos_ids = eos if isinstance(eos, list) else [eos]
if not any(token in eos_ids for token in new_tokens[0].tolist()):
    print("Warning: no EOS; generation may be truncated. Do not treat this as complete.")


## Next steps
For batches, keywords and a command-line script with input validation, see [the runnable example](https://github.com/QwenAudio/Fun-ASR/tree/main/examples/transformers). The native export returns text, not word timestamps, speaker identities, or a realtime protocol. The 31-language MLT checkpoint is separate. CPU success is not a GPU or throughput benchmark; model output still needs human evaluation.

Choose [FunASR services, vLLM or llama.cpp](https://www.funasr.com/en/deploy/) when you need deployment rather than a notebook. Preserve the checkpoint/runtime versions and raw results when evaluating your application.
